# 2 · When the future is announced: perfect-foresight transitions

**The story.** Suppose the island's engineers announce today that productivity will jump next period, jump again the period after, and then slowly fade back to normal. Because the news is *announced and believed*, everyone plans for it in advance — economists call this **perfect foresight**. Question one: how does the economy adjust as the promised boom approaches and passes?

Question two: an earthquake destroys 40% of all machines. How does the island rebuild?

**What's different from notebook 1.** Notebook 1 studied *small* wobbles using a linear approximate copy of the model. Here the moves are large, so we let Dyno solve the *exact*, nonlinear equations across 50 periods at once. The resulting table of values is the **transition path** — a complete script of the future.

### The model's cast and its everyday meaning

| Symbol | Everyday meaning |
| --- | --- |
| `x` | How efficient technology is — given from outside, one number per period, announced in advance |
| `k` | Machines available (capital stock) |
| `c` | Goods consumed now |

A household chooses how much to `c`onsume and how much to rebuild as `k`, period after period. Parameters: `alph` is the share of production credited to machines, `gam` the dislike of uneven consumption over time, `delt` machine wear and tear, `bet` the patience (discount) rate, `aa` the baseline productivity scale, and `T` how many periods we compute.

The model lives in [`models/ramsey_deterministic.dyno`](models/ramsey_deterministic.dyno). It has no random shocks — only an announced productivity plan — so Dyno automatically treats it as deterministic.

In [1]:
import numpy as np
import pandas as pd
from dyno import DynoModel

pd.set_option("display.precision", 4)

## 1. Read the announced plan

Inside [`models/ramsey_deterministic.dyno`](models/ramsey_deterministic.dyno), the productivity path is declared directly:

```text
# Periods 1 and 2 experience an anticipated temporary boost in TFP:
x[1] <- 1.10
x[2] <- 1.30

# Subsequent decay back towards steady-state productivity (x = 1.0):
forall t, 3 <= t < T : x[t] <- 1.0 + (1.30 - 1.0) * exp(-(t - 1))
```

The file states the productivity path, period by period: 1.10 in period 1, 1.30 in period 2, then an exponential decay back toward 1.0 (its resting level) — a `forall` expression fills the later dates up to horizon `T`. Dyno's rendering below confirms what the file says.

In [2]:
model = DynoModel("models/ramsey_deterministic.dyno")
model

Component,Count,Symbols
Equations,2,
Endogenous,2,"k, c"
Exogenous,1,x
Parameters,6,"alph, gam, delt, bet, aa, T"


## 2. Solve every date at once

A *recursive* rule would move period by period using only the past — but people here also react to what is *coming*, so today depends on tomorrow. Dyno therefore stacks all 50 copies of the equations into one giant system and solves them jointly with Newton's method: guess the whole road, measure how much every equation disagrees, correct, repeat. One agreement of all equations is the transition path.

`model.simulate()` returns a `TransitionSimulation`; its `attrs` hold the convergence diagnostics, and `to_df()` provides the table.

In [4]:
transition = model.simulate()
traj_df = transition.to_df()
print("Converged:", transition.attrs["converged"])
print("Newton iterations:", transition.attrs["iterations"])
print("Maximum residual:", f"{transition.attrs['residual']:.2e}")
assert transition.attrs["converged"]
traj_df[["t", "x", "k", "c"]].head(10)

Converged: True
Newton iterations: 3
Maximum residual: 2.56e-12


,t,x,k,c
0,0,1.0000,12.9499,1.5799
1,1,1.1000,13.1166,1.5971
2,2,1.3000,13.5995,1.6606
3,3,1.0406,13.6243,1.6641
4,4,1.0149,13.6044,1.6617
5,5,1.0055,13.5700,1.6575
6,6,1.0020,13.5320,1.6528
7,7,1.0007,13.4945,1.6481
8,8,1.0003,13.4586,1.6436
9,9,1.0001,13.4249,1.6394


`transition.plot(...)` draws the built-in interactive chart. We plot interior dates through period 49: the final date is pinned by the finite-horizon terminal condition (with a horizon you must state what happens at the end — here, the economy is anchored to its resting position).

In [4]:
transition.plot(variables=["x", "k", "c"], T=50, units="level")

alt.Chart(...)

**What the solved path shows.** Capital (`k`) rises while productivity is high and peaks near the largest productivity increase: knowing the boom is coming, people save and build *ahead* of it. Consumption (`c`) rises too and stays above rest long after — enjoying the extra machines the boom paid for. Do not guess the exact timing from the announced numbers: the computed path comes from solving the model's equations together, which is precisely what Dyno did.

## 3. And then a disaster: machines are destroyed

Now hold technology at its resting level 1 forever and study a very different question: a fraction of all machines is destroyed before history starts, and we watch the island rebuild. Write that fraction as a **parameter**, `disaster`, so the model can express *any* size of disaster — and, in the next step, be swept across sizes in one go. (A loss this large is far beyond the linear copy of notebook 1: gaps of ±20% to ±60% are exactly where a linear approximation misjudges.)

Writing the model inline mirrors the prototype recipe of notebook 1 — same ingredients, a different story:

In [5]:
disaster_model_txt = """
# Same economy as models/ramsey_deterministic.dyno, with two changes:
# no productivity news ever, and history starts with fewer machines.
alph     <- 0.50    # share of production credited to machines
gam      <- 0.50    # dislike of uneven consumption over time
delt     <- 0.02    # fraction of machines lost to wear each period
bet      <- 0.051   # patience (discount) rate
aa       <- 0.511   # baseline productivity scale
disaster <- 0.40    # fraction of machines destroyed before history starts
T        <- 50      # periods to compute

# Resting position (steady state)
x[~] <- 1.0
k[~] <- ((delt + bet) / (1.0 * aa * alph))^(1 / (alph - 1))
c[~] <- aa * k[~]^alph - delt * k[~]

# The same two equations as the file: goods and the consumption choice
0 = c[t] + k[t] - aa*x[t]*k[t-1]^alph - (1 - delt)*k[t-1]
0 = c[t]^(-gam) - (1 + bet)^(-1) * (aa*alph*x[t+1]*k[t]^(alph-1) + 1 - delt) * c[t+1]^(-gam)

# No technological news
x[1] <- 1.0
forall t, 2 <= t < T : x[t] <- 1.0

# The disaster: start with fewer machines than at rest
k[0] <- (1 - disaster) * k[~]
"""
disaster_model = DynoModel(txt=disaster_model_txt)
recovery = disaster_model.solve()
assert recovery.attrs["converged"]

recovery.plot(variables=["k", "c"], T=50, units="percent")
print(f"Capital gap in period 0: {100 * (recovery.to_df().loc[0, 'k'] / disaster_model.steady_state['k'] - 1):.1f}%")
print(f"Capital gap after 20 periods: {100 * (recovery.to_df().loc[20, 'k'] / disaster_model.steady_state['k'] - 1):.1f}%")


Capital gap in period 0: -40.0%
Capital gap after 20 periods: -11.2%


### How bad is bad? Sweep the disaster size

One parameter, one sweep: `model.variants(disaster=[...])` builds a recalibrated model per fraction destroyed — the same chained pipeline used for the islands in notebook 1. The table and chart compare the rebuild under three increasingly severe hits. Notice the pattern: recovery scales **almost proportionally** to the size of the hit. Destroy 20% and consumption dips 19%; triple the destruction to 60% and the dip roughly triples to 58%. This model's mathematics is *homogeneous* — rescaling the machines rescales the response — so one recovery curve almost determines the others; the chart confirms the near-copies.

In [6]:
recoveries = disaster_model.variants(disaster=[0.20, 0.40, 0.60]).solve()

comparison = (
    recoveries.to_df(units="percent")
    .groupby("variant", sort=False)
    .agg({"k": "first", "c": "min"})
)
comparison.columns = ["Capital gap, period 0 (%)", "Worst drop in consumption (%)"]
display(comparison)

recoveries.plot(variables=["k", "c"], T=50, units="percent")

,"Capital gap, period 0 (%)",Worst drop in consumption (%)
variant,,
disaster=0.2,-20.0,-19.2140
disaster=0.4,-40.0,-38.5588
disaster=0.6,-60.0,-58.1054


alt.Chart(...)

Consumption pays for the rebuild in every scenario: it falls almost as far as capital right after the disaster (−38.6% in period 1 for the 40% case), then climbs back steadily as the machines return — a slow, smooth recovery.

**Takeaway.** Deterministic transitions answer questions about an *announced* path or a *large* starting dislocation, solved exactly rather than linearly. Parameters make every part of the question adjustable, and `variants` sweeps those parameters across recalibrated models — above, one line produced the three recovery curves.